# Task 3 — Colab notebook для эксперта-создателя набора

Этот notebook нужен **эксперту-создателю тестового набора** для Task 3.

## Зачем нужна работа эксперта-создателя

В проекте `top-papers-graph` Task 3 отвечает за генерацию гипотез на основе графа знаний и мультимодальных evidence из научных публикаций. Чтобы честно проверить, где **baseline VLM** и **SFT/DPO VLM** действительно отличаются, нужен **качественно собранный набор кейсов**, а не случайная выборка.

Роль эксперта-создателя:
- выбрать и описать **диагностические кейсы**;
- сместить набор в сторону **multimodal_hard** и **temporal_hard**;
- задать понятные критерии, по которым потом **эксперт-участник** сможет сравнивать варианты A/B;
- подготовить **offline-форму**, которую удобно заполнять без ручной правки JSON.

В конце работы у вас будут готовы:
1. `task1_minimal_for_ab_bundle.zip`
2. `task3_ab_creator_bundle_ru.zip`
3. `task3_ab_case_manifest.template.json`
4. `task3_ab_creator_offline_form_ru.html`


## Почему форма устроена именно так

Форма ниже специально переделана под удобную экспертную работу:

- у каждого поля есть **явный заголовок** и **подсказка рядом с полем**, а не только placeholder;
- сложные технические поля сопоставления (`match.*`) убраны в **раскрывающийся блок**, чтобы не перегружать экран;
- форма показывает **статус готовности кейса** и список того, чего не хватает;
- значения **автосохраняются** в браузере;
- есть кнопки **«Добавить multimodal_hard / temporal_hard / easy_control»** и **«Дублировать кейс»**, чтобы быстрее собирать набор;
- поля `review_focus` и `expected_error_modes` сделаны как **чекбоксы**, а не как ручной ввод через запятую.

Такой подход уменьшает когнитивную нагрузку и снижает вероятность ошибок при заполнении сложных форм. Явные labels и helper text обычно улучшают понятность полей, а progressive disclosure помогает скрывать сложность до нужного момента. citeturn646465search1turn646465search2turn646465search4turn646465search13


In [ ]:

# @title Шаг 1. Установка зависимостей и загрузка репозитория
!pip -q install pyyaml ipywidgets nbformat

import datetime as dt
import hashlib
import json
import re
import sys
import unicodedata
import zipfile
from pathlib import Path

import yaml
from IPython.display import display, Markdown, FileLink, IFrame

REPO_DIR = Path('/content/top-papers-graph')
REPO_URL = 'https://github.com/top-papers/top-papers-graph.git'

if not REPO_DIR.exists():
    !git clone --depth 1 {REPO_URL} {REPO_DIR}

if str(REPO_DIR / 'src') not in sys.path:
    sys.path.insert(0, str(REPO_DIR / 'src'))

from scireason.task3_ab_case_manifest import default_case_manifest

CURRENT_YEAR = dt.datetime.utcnow().year

def _slugify(text: str) -> str:
    text = str(text or '').strip().lower()
    text = unicodedata.normalize('NFKD', text)
    text = ''.join(ch for ch in text if not unicodedata.combining(ch))
    text = re.sub(r'[^a-z0-9а-яё]+', '-', text, flags=re.IGNORECASE)
    text = re.sub(r'-{2,}', '-', text).strip('-')
    return text or 'submission'

def _short_hash(text: str) -> str:
    return hashlib.sha256(str(text).encode('utf-8')).hexdigest()[:10]

def build_minimal_task1_yaml(*, topic: str, cutoff_year: int, identifiers_text: str, expert_name: str = '') -> dict:
    papers = []
    for raw in re.split(r'[\n;,]+', identifiers_text or ''):
        raw = raw.strip()
        if not raw:
            continue
        papers.append({'id': raw, 'title': '', 'year': 0})
    submission_seed = topic or identifiers_text or 'task3-ab'
    expert_name = (expert_name or '').strip()
    return {
        'submission_id': f"{_slugify(expert_name or 'expert')}__{_short_hash(submission_seed)}",
        'topic': topic.strip(),
        'cutoff_year': int(cutoff_year),
        'domain': 'science',
        'expert': {'full_name': expert_name, 'latin_slug': _slugify(expert_name or 'expert')},
        'papers': papers,
        'steps': [],
        'edges': [],
    }

def write_task1_bundle(payload: dict, out_dir: Path) -> Path:
    out_dir.mkdir(parents=True, exist_ok=True)
    yaml_path = out_dir / 'task1_minimal_for_ab.yaml'
    yaml_path.write_text(yaml.safe_dump(payload, allow_unicode=True, sort_keys=False), encoding='utf-8')
    (out_dir / 'manifest.json').write_text(
        json.dumps({'submission_id': payload.get('submission_id'), 'topic': payload.get('topic')}, ensure_ascii=False, indent=2),
        encoding='utf-8'
    )
    (out_dir / 'README.txt').write_text(
        'Загрузите task1_minimal_for_ab.yaml или ZIP-бандл в Task 3 runner.\n',
        encoding='utf-8'
    )
    zip_path = out_dir / 'task1_minimal_for_ab_bundle.zip'
    with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
        for p in [yaml_path, out_dir/'manifest.json', out_dir/'README.txt']:
            zf.write(p, arcname=p.name)
    return zip_path


## Шаг 2. Заполните параметры

Ниже укажите:
- имя эксперта;
- тему;
- верхнюю границу по году;
- список идентификаторов статей.

Подойдут DOI, arXiv ID, PMID, локальные `paper_id` или просто список ключевых идентификаторов, которые вы будете использовать дальше.


In [ ]:

# @title Параметры создателя набора
EXPERT_NAME = "Имя Фамилия"
TOPIC = "Temporal catalyst latency forecasting"
CUTOFF_YEAR = CURRENT_YEAR
IDENTIFIERS_TEXT = """
10.1000/example-doi-001
arXiv:2401.01234
pmid:12345678
"""
DEFAULT_CASE_COUNT = 12
OUTPUT_DIR = Path('/content/task3_ab_creator_outputs')


## Шаг 3. Соберите минимальный trajectory bundle

Этот шаг создаёт минимальный Task 1 YAML/ZIP. Он нужен как связующий артефакт для дальнейшего Task 3 пайплайна.

Что вы получите:
- `task1_minimal_for_ab.yaml`
- `task1_minimal_for_ab_bundle.zip`


In [ ]:

# @title Создать минимальный trajectory bundle
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

trajectory_payload = build_minimal_task1_yaml(
    topic=TOPIC,
    cutoff_year=CUTOFF_YEAR,
    identifiers_text=IDENTIFIERS_TEXT,
    expert_name=EXPERT_NAME,
)

trajectory_zip = write_task1_bundle(trajectory_payload, OUTPUT_DIR / 'trajectory_bundle')

display(Markdown("### Готово"))
print("submission_id:", trajectory_payload["submission_id"])
print("trajectory bundle:", trajectory_zip)
display(FileLink(str(trajectory_zip)))


## Шаг 4. Соберите русскую офлайн-форму с UX-подсказками

Сейчас notebook создаст:
- русский HTML-файл для заполнения кейсов;
- JSON-шаблон;
- краткую памятку;
- ZIP-бандл для передачи или скачивания.

### Что будет удобно в форме
- карточки кейсов с понятными секциями;
- встроенные примеры и пояснения;
- автоматические статусы «готов / не готов»;
- advanced-блок для `match.*`;
- автосохранение в браузере;
- **скачивание черновика** и **загрузка черновика обратно в форму**;
- заметную кнопку **перехода в Google Форму** для отправки данных:  
  `https://forms.gle/h5RwEA8DsZh9pBAt8`
- экспорт заполненного JSON.


In [ ]:

# @title Создать русскую форму и creator bundle
def build_creator_offline_form_ru(*, output_dir: Path, seed_manifest: dict) -> dict:
    output_dir.mkdir(parents=True, exist_ok=True)

    manifest_path = output_dir / 'task3_ab_case_manifest.template.json'
    manifest_path.write_text(json.dumps(seed_manifest, ensure_ascii=False, indent=2), encoding='utf-8')

    tutorial_text = """# Как заполнять офлайн-форму Task 3 для создателя набора

1. Откройте `task3_ab_creator_offline_form_ru.html` в браузере.
2. Сначала заполните блок **«Метаданные эксперимента»**.
3. Затем по каждому кейсу:
   - укажите статью и тип evidence;
   - сформулируйте, на что должен смотреть участник A/B;
   - выберите зоны внимания (`review_focus`);
   - отметьте ожидаемые типы ошибок;
   - при необходимости раскройте блок **«Техническое сопоставление»** и заполните `match.*`.
4. Если хотите сделать паузу, нажмите **«Скачать черновик JSON»**.
5. Чтобы продолжить позже, используйте **«Загрузить черновик JSON»**.
6. Следите за индикатором готовности кейса.
7. Нажмите **«Скачать заполненный JSON»** и сохраните `task3_ab_case_manifest.filled.json`.
8. Если у вас принят процесс сбора через веб, нажмите кнопку **«Открыть Google Форму для отправки данных»** и перейдите по ссылке:
   `https://forms.gle/h5RwEA8DsZh9pBAt8`
"""

    (output_dir / 'README_RU.txt').write_text(
        "1) Откройте task3_ab_creator_offline_form_ru.html в браузере.\n"
        "2) Заполните кейсы на русском языке.\n"
        "3) При необходимости скачайте черновик JSON или загрузите ранее сохранённый черновик.\n"
        "4) Скачайте task3_ab_case_manifest.filled.json.\n"
        "5) При вашем процессе сбора перейдите по ссылке Google Формы: https://forms.gle/h5RwEA8DsZh9pBAt8\n"
        "6) Используйте этот JSON как case manifest в CLI/Kaggle/DataSphere runner.\n",
        encoding='utf-8',
    )
    (output_dir / 'TASK3_AB_CREATOR_FORM_TUTORIAL_RU.md').write_text(tutorial_text, encoding='utf-8')
    (output_dir / 'task3_ab_creator_checklist_ru.md').write_text(
        "# Чек-лист создателя набора\n\n"
        "- у каждого кейса заполнен paper_title;\n"
        "- у каждого кейса есть creator_prompt и creator_rationale;\n"
        "- у каждого кейса заполнено хотя бы одно match-поле;\n"
        "- easy_control — меньшинство;\n"
        "- expected winner не раскрыт;\n"
        "- перед экспортом проверены предупреждения формы.\n",
        encoding='utf-8',
    )

    html_template = r'''<!doctype html>
<html lang="ru">
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>Task 3 — офлайн-форма создателя набора</title>
<style>
:root{
  --bg:#f6f8fb; --card:#ffffff; --text:#0f172a; --muted:#475569; --line:#dbe3ef;
  --blue:#2563eb; --bluebg:#eff6ff; --green:#166534; --greenbg:#ecfdf5; --amber:#a16207; --amberbg:#fefce8; --red:#b91c1c; --redbg:#fef2f2;
}
*{box-sizing:border-box}
body{margin:0;font-family:Inter,system-ui,-apple-system,sans-serif;background:var(--bg);color:var(--text)}
.page{max-width:1280px;margin:0 auto;padding:24px}
.hero,.card,.case{background:var(--card);border:1px solid var(--line);border-radius:18px;padding:18px;box-shadow:0 10px 24px rgba(15,23,42,.05)}
.hero,.card{margin-bottom:16px}
.case{margin-top:14px}
h1,h2,h3{margin:0 0 10px 0}
p{margin:8px 0}
.muted{color:var(--muted)}
.grid{display:grid;gap:12px}
.cols2{grid-template-columns:repeat(auto-fit,minmax(280px,1fr))}
.cols3{grid-template-columns:repeat(auto-fit,minmax(220px,1fr))}
.toolbar{display:flex;gap:8px;flex-wrap:wrap;align-items:center}
button,.buttonlike{border:1px solid #cbd5e1;border-radius:12px;padding:10px 14px;background:#fff;cursor:pointer;font-weight:600}
button.primary{background:var(--blue);color:#fff;border-color:var(--blue)}
input,textarea,select{width:100%;padding:10px 12px;border:1px solid #cbd5e1;border-radius:12px;font:inherit;background:#fff}
textarea{min-height:88px;resize:vertical}
label.field{display:block}
label.field .title{display:block;font-weight:700;margin-bottom:6px}
label.field .help{display:block;color:var(--muted);font-size:13px;line-height:1.4;margin-top:6px}
.section-title{font-size:15px;font-weight:800;margin:10px 0 8px}
.pills{display:flex;gap:8px;flex-wrap:wrap}
.pill{display:inline-flex;align-items:center;padding:6px 10px;border-radius:999px;background:var(--bluebg);color:var(--blue);font-size:12px;font-weight:700}
.pill.ok{background:var(--greenbg);color:var(--green)}
.pill.warn{background:var(--amberbg);color:var(--amber)}
.pill.bad{background:var(--redbg);color:var(--red)}
.checkgrid{display:grid;grid-template-columns:repeat(auto-fit,minmax(220px,1fr));gap:8px}
.check{display:flex;gap:8px;align-items:flex-start;border:1px solid var(--line);border-radius:12px;padding:10px;background:#fcfdff}
.check input{width:auto;margin-top:3px}
details{border:1px dashed #cbd5e1;border-radius:14px;padding:12px;background:#fbfdff}
summary{cursor:pointer;font-weight:700}
.notice{border-left:4px solid var(--blue);background:#f8fbff;padding:12px 14px;border-radius:12px}
.casehead{display:flex;justify-content:space-between;gap:12px;flex-wrap:wrap;align-items:flex-start}
.case-actions{display:flex;gap:8px;flex-wrap:wrap}
.small{font-size:12px}
hr.sep{border:none;border-top:1px solid var(--line);margin:16px 0}
</style>
</head>
<body>
<div class="page">
  <div class="hero">
    <h1>Task 3 — форма создателя набора A/B</h1>
    <p class="muted">Эту форму заполняет эксперт-создатель. Её задача — подготовить диагностические кейсы, на которых различия между baseline VLM и SFT/DPO VLM будут видны именно там, где это важно для Task 3: в multimodal evidence extraction, evidence linkage и temporal grounding.</p>
    <div class="notice">
      <b>Как пользоваться формой</b><br>
      1) заполните метаданные эксперимента; 2) добавьте кейсы по нужным стратам; 3) при необходимости скачайте черновик и продолжите позже через загрузку JSON; 4) для каждого кейса кратко сформулируйте, что должен оценивать участник A/B; 5) при необходимости раскройте технический блок сопоставления; 6) скачайте заполненный JSON или перейдите в Google Форму для отправки данных.
    </div>
    <div class="toolbar" style="margin-top:12px">
      <button class="primary" id="exportFilled">Скачать заполненный JSON</button>
      <button id="exportDraft">Скачать черновик JSON</button>
      <label class="buttonlike" for="loadDraft">Загрузить черновик JSON</label>
      <input id="loadDraft" type="file" accept="application/json" style="display:none">
      <button id="clearLocalDraft">Очистить локальный черновик</button>
      <a class="buttonlike" id="openGoogleForm" href="https://forms.gle/h5RwEA8DsZh9pBAt8" target="_blank" rel="noopener noreferrer">Открыть Google Форму для отправки данных</a>
    </div>
    <div class="notice" style="margin-top:12px">
      <b>Работа с черновиком</b><br>
      Нажмите <b>«Скачать черновик JSON»</b>, если хотите продолжить позже или передать незавершённый вариант коллеге. Чтобы восстановить работу, используйте <b>«Загрузить черновик JSON»</b>. Для быстрой отправки данных в веб-канал используйте кнопку <b>«Открыть Google Форму для отправки данных»</b>.
    </div>
    <div class="pills" id="stats" style="margin-top:12px"></div>
  </div>

  <div class="card">
    <h2>1. Метаданные эксперимента</h2>
    <p class="muted">Сначала зафиксируйте тему, submission_id и цель проверки. Эти поля попадут в итоговый manifest.</p>
    <div class="grid cols3" id="meta"></div>
  </div>

  <div class="card">
    <h2>2. Кейсы для набора</h2>
    <p class="muted">Рекомендуемый баланс: больше <b>multimodal_hard</b> и <b>temporal_hard</b>, меньше <b>easy_control</b>.</p>
    <div class="toolbar">
      <button id="addMM">Добавить multimodal_hard</button>
      <button id="addTemporal">Добавить temporal_hard</button>
      <button id="addControl">Добавить easy_control</button>
    </div>
    <div id="cases"></div>
  </div>
</div>

<script>
const APP = __APP_DATA__;
const STORAGE_KEY = `task3-authoring-ru:${(APP.experiment_meta||{}).submission_id||'draft'}`;

const STRATUM_LABELS = {
  multimodal_hard: "multimodal_hard — важный факт сидит в figure/table/page",
  temporal_hard: "temporal_hard — важны временные рамки, динамика, before/after",
  easy_control: "easy_control — простой контрольный кейс"
};
const EVIDENCE_KIND_OPTIONS = [
  ["figure","figure"],
  ["table","table"],
  ["figure_or_table","figure_or_table"],
  ["formula","formula"],
  ["page","page"],
  ["mixed","mixed"]
];
const REVIEW_FOCUS_OPTIONS = [
  ["evidence","Извлечение evidence"],
  ["visual_fact","Визуальный факт / figure/table-driven факт"],
  ["temporal","Временная привязка и temporal scope"],
  ["overall","Общее качество кейса"]
];
const ERROR_OPTIONS = [
  ["missed_visual_fact","Пропущен важный visual fact"],
  ["wrong_evidence_linkage","Неверная привязка evidence"],
  ["needs_time_fix","Ошибка во временной привязке"],
  ["hallucinated_visual_inference","Придуман visual inference"]
];
const FIELD_HELP = {
  topic: "Например: Temporal catalyst latency forecasting",
  submission_id: "Заполняется автоматически, но можно скорректировать вручную",
  creator_id: "Имя или идентификатор эксперта-создателя",
  cutoff_year: "Верхняя граница по году публикаций",
  review_goal: "Коротко: что именно должен показать A/B тест",
  case_id: "Уникальный ID кейса, например CASE-001",
  paper_title: "Полное или сокращённое название статьи",
  paper_id: "DOI / arXiv / PMID / локальный paper_id",
  year: "Год статьи",
  evidence_kind: "Где сидит ключевой факт: figure, table, page, formula...",
  page_hint: "Например: p.4, Fig.2 или Table 3",
  creator_prompt: "Коротко и по делу: на что должен смотреть эксперт-участник",
  creator_rationale: "Почему этот кейс диагностический именно для VLM-разницы",
  notes: "Необязательные внутренние заметки создателя",
  candidate_signature: "Если сигнатура известна и стабильна — это самый точный способ сопоставления",
  candidate_source_contains: "Фрагмент source из candidate",
  candidate_predicate_contains: "Фрагмент predicate из candidate",
  candidate_target_contains: "Фрагмент target из candidate",
  hypothesis_title_contains: "Подстрока из title гипотезы",
  premise_contains: "Подстрока из premise",
  mechanism_contains: "Подстрока из mechanism",
  time_scope_contains: "Подстрока из time_scope",
  rank_hint: "Ожидаемый rank, если он полезен как дополнительная подсказка"
};

const el = (tag, attrs, ...children) => {
  const node = document.createElement(tag);
  if (attrs) {
    for (const [k,v] of Object.entries(attrs)) {
      if (v == null) continue;
      if (k === "class") node.className = v;
      else if (k === "text") node.textContent = v;
      else if (k === "html") node.innerHTML = v;
      else if (k.startsWith("on")) node.addEventListener(k.slice(2), v);
      else node.setAttribute(k, v);
    }
  }
  children.flat().forEach(ch => {
    if (ch == null) return;
    node.appendChild(typeof ch === "string" ? document.createTextNode(ch) : ch);
  });
  return node;
};

const state = JSON.parse(JSON.stringify(APP));

function save() {
  try { localStorage.setItem(STORAGE_KEY, JSON.stringify(state)); } catch (_) {}
}
function load() {
  try {
    const raw = localStorage.getItem(STORAGE_KEY);
    if (raw) Object.assign(state, JSON.parse(raw));
  } catch (_) {}
}
function download(name, text, mime='application/json') {
  const blob = new Blob([text], {type:mime});
  const url = URL.createObjectURL(blob);
  const a = document.createElement('a');
  a.href = url; a.download = name;
  document.body.appendChild(a); a.click(); a.remove();
  setTimeout(() => URL.revokeObjectURL(url), 1000);
}
function field(label, help, inputNode) {
  return el('label', {class:'field'},
    el('span', {class:'title', text:label}),
    inputNode,
    help ? el('span', {class:'help', text:help}) : null
  );
}
function norm(v){ return String(v||'').trim(); }
function requiredMissing(caseObj) {
  const miss = [];
  if (!norm(caseObj.paper_title)) miss.push("paper_title");
  if (!norm(caseObj.creator_prompt)) miss.push("creator_prompt");
  if (!norm(caseObj.creator_rationale)) miss.push("creator_rationale");
  if (!Array.isArray(caseObj.review_focus) || !caseObj.review_focus.length) miss.push("review_focus");
  const m = caseObj.match || {};
  const anyMatch = ["candidate_signature","candidate_source_contains","candidate_predicate_contains","candidate_target_contains","hypothesis_title_contains","premise_contains","mechanism_contains","time_scope_contains","rank_hint"].some(k => norm(m[k]));
  if (!anyMatch) miss.push("минимум одно поле match.*");
  return miss;
}
function caseStatus(caseObj) {
  const miss = requiredMissing(caseObj);
  if (!miss.length) return {label:"Готов к экспорту", cls:"ok"};
  if (miss.length <= 2) return {label:`Нужно дополнить: ${miss.join(", ")}`, cls:"warn"};
  return {label:`Не готов: ${miss.join(", ")}`, cls:"bad"};
}
function renderStats() {
  const cases = (state.cases || []).filter(c => c.enabled !== false);
  const ready = cases.filter(c => !requiredMissing(c).length).length;
  const by = {multimodal_hard:0, temporal_hard:0, easy_control:0};
  cases.forEach(c => by[c.stratum] = (by[c.stratum] || 0) + 1);
  const host = document.getElementById('stats');
  host.innerHTML = '';
  host.append(
    el('span', {class:'pill', text:`Всего кейсов: ${cases.length}`}),
    el('span', {class:'pill ok', text:`Готовы: ${ready}`}),
    el('span', {class:'pill', text:`MM: ${by.multimodal_hard||0}`}),
    el('span', {class:'pill', text:`Temporal: ${by.temporal_hard||0}`}),
    el('span', {class:'pill', text:`Control: ${by.easy_control||0}`}),
  );
}
function renderMeta() {
  const meta = state.experiment_meta || (state.experiment_meta = {});
  const host = document.getElementById('meta');
  host.innerHTML = '';
  [["topic","Тема"],["submission_id","submission_id"],["creator_id","creator_id"],["cutoff_year","cutoff_year"],["review_goal","Цель проверки"]]
    .forEach(([key,label]) => {
      const node = key === "review_goal" ? el('textarea', {}) : el('input', {type:'text'});
      node.value = meta[key] || '';
      node.oninput = () => { meta[key] = node.value; save(); renderStats(); };
      host.append(field(label, FIELD_HELP[key], node));
    });
}
function setPreset(caseObj, stratum) {
  caseObj.stratum = stratum;
  if (stratum === 'multimodal_hard') {
    caseObj.review_focus = ['evidence','visual_fact'];
    caseObj.expected_error_modes = ['missed_visual_fact','wrong_evidence_linkage'];
    caseObj.evidence_kind = caseObj.evidence_kind || 'figure_or_table';
  } else if (stratum === 'temporal_hard') {
    caseObj.review_focus = ['temporal','evidence'];
    caseObj.expected_error_modes = ['needs_time_fix','wrong_evidence_linkage'];
    caseObj.evidence_kind = caseObj.evidence_kind || 'figure_or_table';
  } else {
    caseObj.review_focus = ['overall'];
    caseObj.expected_error_modes = [];
    caseObj.evidence_kind = caseObj.evidence_kind || 'page';
  }
}
function checkboxGroup(options, selected, onChange) {
  const set = new Set(selected || []);
  const box = el('div', {class:'checkgrid'});
  options.forEach(([value,label]) => {
    const input = el('input', {type:'checkbox'});
    input.checked = set.has(value);
    input.onchange = () => {
      if (input.checked) set.add(value); else set.delete(value);
      onChange(Array.from(set));
    };
    box.append(el('label', {class:'check'}, input, el('div', null, el('div', {text:label}), el('div', {class:'small muted', text:value}))));
  });
  return box;
}
function addCase(stratum='multimodal_hard') {
  const idx = (state.cases || []).length + 1;
  const c = {
    case_id: `CASE-${String(idx).padStart(3,'0')}`,
    enabled: true,
    primary_endpoint: stratum !== 'easy_control',
    stratum,
    paper_title: '',
    paper_id: '',
    year: '',
    evidence_kind: '',
    page_hint: '',
    creator_prompt: '',
    creator_rationale: '',
    review_focus: [],
    expected_error_modes: [],
    match: {
      candidate_signature: '',
      candidate_source_contains: '',
      candidate_predicate_contains: '',
      candidate_target_contains: '',
      hypothesis_title_contains: '',
      premise_contains: '',
      mechanism_contains: '',
      time_scope_contains: '',
      rank_hint: '',
    },
    notes: '',
  };
  setPreset(c, stratum);
  (state.cases || (state.cases = [])).push(c);
  render();
  save();
}
function cloneCase(idx) {
  const original = state.cases[idx];
  const cloned = JSON.parse(JSON.stringify(original));
  cloned.case_id = `CASE-${String((state.cases||[]).length + 1).padStart(3,'0')}`;
  state.cases.push(cloned);
  render();
  save();
}
function selectStratum(c) {
  const s = el('select', null,
    ['multimodal_hard','temporal_hard','easy_control'].map(v => el('option', {value:v, text:STRATUM_LABELS[v]}))
  );
  s.value = c.stratum || 'multimodal_hard';
  s.onchange = () => { setPreset(c, s.value); render(); save(); };
  return s;
}
function selectEvidenceKind(c) {
  const s = el('select', null, EVIDENCE_KIND_OPTIONS.map(([v,label]) => el('option', {value:v, text:label})));
  s.value = c.evidence_kind || 'figure_or_table';
  s.onchange = () => { c.evidence_kind = s.value; save(); };
  return s;
}
function caseCard(c, idx) {
  const status = caseStatus(c);
  const box = el('div', {class:'case'});
  const enabled = el('input', {type:'checkbox'});
  enabled.checked = c.enabled !== false;
  enabled.onchange = () => { c.enabled = enabled.checked; save(); renderStats(); };
  const primary = el('input', {type:'checkbox'});
  primary.checked = !!c.primary_endpoint;
  primary.onchange = () => { c.primary_endpoint = primary.checked; save(); };

  box.append(
    el('div', {class:'casehead'},
      el('div', null,
        el('div', {class:'pills'},
          el('span', {class:'pill', text:c.case_id || `CASE-${String(idx+1).padStart(3,'0')}`}),
          el('span', {class:`pill ${status.cls}`, text:status.label})
        ),
        el('p', {class:'muted small', text:'Сначала заполните контекст кейса, затем задайте понятный creator_prompt, а технические match-поля раскрывайте только при необходимости.'})
      ),
      el('div', {class:'case-actions'},
        el('label', {class:'small'}, enabled, document.createTextNode(' включён')),
        el('label', {class:'small'}, primary, document.createTextNode(' primary endpoint')),
        el('button', {text:'Дублировать кейс', onClick:() => cloneCase(idx)}),
        el('button', {text:'Удалить кейс', onClick:() => { state.cases.splice(idx, 1); render(); save(); }})
      )
    )
  );

  box.append(el('div', {class:'section-title', text:'A. Контекст кейса'}));
  const top = el('div', {class:'grid cols3'});
  const caseIdInput = el('input', {type:'text', value:c.case_id || ''});
  caseIdInput.oninput = () => { c.case_id = caseIdInput.value; save(); renderStats(); };
  const paperTitle = el('input', {type:'text', value:c.paper_title || '', placeholder:'Например: Catalyst latency under periodic modulation'});
  paperTitle.oninput = () => { c.paper_title = paperTitle.value; save(); render(); };
  const paperId = el('input', {type:'text', value:c.paper_id || '', placeholder:'doi:..., arXiv:..., pmid:...'});
  paperId.oninput = () => { c.paper_id = paperId.value; save(); };
  const year = el('input', {type:'text', value:c.year || '', placeholder:'2024'});
  year.oninput = () => { c.year = year.value; save(); };
  const pageHint = el('input', {type:'text', value:c.page_hint || '', placeholder:'p.4, Fig.2 / Table 3'});
  pageHint.oninput = () => { c.page_hint = pageHint.value; save(); };
  top.append(
    field('case_id', FIELD_HELP.case_id, caseIdInput),
    field('paper_title', FIELD_HELP.paper_title, paperTitle),
    field('paper_id', FIELD_HELP.paper_id, paperId),
    field('year', FIELD_HELP.year, year),
    field('evidence_kind', FIELD_HELP.evidence_kind, selectEvidenceKind(c)),
    field('page_hint', FIELD_HELP.page_hint, pageHint),
    field('stratum', 'Выберите тип сложности кейса. Это влияет на рекомендуемые focus/error presets.', selectStratum(c))
  );
  box.append(top);

  box.append(el('hr', {class:'sep'}));
  box.append(el('div', {class:'section-title', text:'B. Что должен оценивать участник A/B'}));
  const prompt = el('textarea', {placeholder:'Пример: Сравните, какой вариант лучше извлекает численные значения из таблицы 3 и не теряет их в hypothesis support.'});
  prompt.value = c.creator_prompt || '';
  prompt.oninput = () => { c.creator_prompt = prompt.value; save(); render(); };
  const rationale = el('textarea', {placeholder:'Пример: baseline часто пропускает колонку с доверительными интервалами, а tuned VLM должен сохранить численные evidence и правильную привязку к таблице.'});
  rationale.value = c.creator_rationale || '';
  rationale.oninput = () => { c.creator_rationale = rationale.value; save(); render(); };
  box.append(field('creator_prompt', FIELD_HELP.creator_prompt, prompt));
  box.append(field('creator_rationale', FIELD_HELP.creator_rationale, rationale));

  box.append(el('hr', {class:'sep'}));
  box.append(el('div', {class:'section-title', text:'C. На что смотреть при ревью'}));
  const focusWrap = el('div');
  focusWrap.append(
    el('div', {class:'muted small', text:'Выберите зоны внимания. Лучше 1–3 пункта, а не всё сразу.'}),
    checkboxGroup(REVIEW_FOCUS_OPTIONS, c.review_focus || [], (vals) => { c.review_focus = vals; save(); render(); })
  );
  const errWrap = el('div');
  errWrap.append(
    el('div', {class:'muted small', text:'Отметьте ожидаемые типы ошибок, которые этот кейс должен уметь вскрывать.'}),
    checkboxGroup(ERROR_OPTIONS, c.expected_error_modes || [], (vals) => { c.expected_error_modes = vals; save(); })
  );
  box.append(
    el('div', {class:'grid cols2'},
      field('review_focus', 'Это то, что участник должен оценить в первую очередь.', focusWrap),
      field('expected_error_modes', 'Это ожидаемые fail modes именно для этого кейса.', errWrap)
    )
  );

  box.append(el('hr', {class:'sep'}));
  const details = el('details', null,
    el('summary', {text:'D. Техническое сопоставление (advanced)'}),
    el('p', {class:'muted small', text:'Раскрывайте этот блок, если нужно точнее привязать кейс к candidate/hypothesis в downstream runner. Для большинства кейсов достаточно 1–2 match-полей.'})
  );
  const m = c.match || (c.match = {});
  const matchGrid = el('div', {class:'grid cols2'});
  [
    ['candidate_signature','candidate_signature'],
    ['candidate_source_contains','candidate_source_contains'],
    ['candidate_predicate_contains','candidate_predicate_contains'],
    ['candidate_target_contains','candidate_target_contains'],
    ['hypothesis_title_contains','hypothesis_title_contains'],
    ['premise_contains','premise_contains'],
    ['mechanism_contains','mechanism_contains'],
    ['time_scope_contains','time_scope_contains'],
    ['rank_hint','rank_hint'],
  ].forEach(([key,label]) => {
    const input = el('input', {type:'text', value:m[key] || ''});
    input.oninput = () => { m[key] = input.value; save(); render(); };
    matchGrid.append(field(label, FIELD_HELP[key], input));
  });
  details.append(matchGrid);
  box.append(details);

  box.append(el('hr', {class:'sep'}));
  const notes = el('textarea', {placeholder:'Необязательные заметки: почему кейс спорный, что проверить вручную, чего избегать.'});
  notes.value = c.notes || '';
  notes.oninput = () => { c.notes = notes.value; save(); };
  box.append(field('notes', FIELD_HELP.notes, notes));

  return box;
}
function renderCases() {
  const host = document.getElementById('cases');
  host.innerHTML = '';
  (state.cases || []).forEach((c, idx) => host.append(caseCard(c, idx)));
}
function render() {
  renderMeta();
  renderCases();
  renderStats();
}
document.getElementById('addMM').onclick = () => addCase('multimodal_hard');
document.getElementById('addTemporal').onclick = () => addCase('temporal_hard');
document.getElementById('addControl').onclick = () => addCase('easy_control');
document.getElementById('exportFilled').onclick = () => download('task3_ab_case_manifest.filled.json', JSON.stringify(state, null, 2));
document.getElementById('exportDraft').onclick = () => download('task3_ab_case_manifest.draft.json', JSON.stringify(state, null, 2));
document.getElementById('clearLocalDraft').onclick = () => {
  if (!confirm('Удалить локально сохранённый черновик из браузера?')) return;
  try { localStorage.removeItem(STORAGE_KEY); } catch (_) {}
  alert('Локальный черновик удалён. Текущие данные на экране не изменены. Чтобы начать с нуля, обновите страницу.');
};
document.getElementById('loadDraft').addEventListener('change', (ev) => {
  const f = ev.target.files && ev.target.files[0];
  if (!f) return;
  const reader = new FileReader();
  reader.onload = () => {
    Object.assign(state, JSON.parse(String(reader.result || '{}')));
    render();
    save();
  };
  reader.readAsText(f, 'utf-8');
});
load();
if (!(state.cases || []).length) {
  addCase('multimodal_hard');
  addCase('temporal_hard');
  addCase('easy_control');
}
render();
save();
</script>
</body>
</html>'''

    app_json = json.dumps(seed_manifest, ensure_ascii=False).replace("</", "<\\/")
    html_path = output_dir / 'task3_ab_creator_offline_form_ru.html'
    html_path.write_text(html_template.replace('__APP_DATA__', app_json), encoding='utf-8')

    bundle_zip = output_dir / 'task3_ab_creator_bundle_ru.zip'
    with zipfile.ZipFile(bundle_zip, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
        for p in [
            manifest_path,
            output_dir / 'README_RU.txt',
            output_dir / 'TASK3_AB_CREATOR_FORM_TUTORIAL_RU.md',
            output_dir / 'task3_ab_creator_checklist_ru.md',
            html_path,
        ]:
            zf.write(p, arcname=p.name)

    return {
        'manifest_template': manifest_path,
        'offline_form_ru': html_path,
        'tutorial_ru': output_dir / 'TASK3_AB_CREATOR_FORM_TUTORIAL_RU.md',
        'checklist_ru': output_dir / 'task3_ab_creator_checklist_ru.md',
        'bundle_zip': bundle_zip,
    }

seed_manifest = default_case_manifest(
    topic=TOPIC,
    submission_id=trajectory_payload['submission_id'],
    creator_id=EXPERT_NAME,
    cutoff_year=str(CUTOFF_YEAR),
    case_count=DEFAULT_CASE_COUNT,
)

authoring_assets = build_creator_offline_form_ru(
    output_dir=OUTPUT_DIR / 'authoring_bundle_ru',
    seed_manifest=seed_manifest,
)

display(Markdown("### Русская форма и bundle готовы"))
for key, value in authoring_assets.items():
    print(f"{key}: {value}")
    display(FileLink(str(value)))


## Шаг 5. Проверьте форму перед передачей эксперту

Откройте HTML прямо в Colab или скачайте его локально.  
Ниже — удобный preview. Если браузер встраивание блокирует, просто скачайте файл через ссылку.


In [ ]:

# @title Предпросмотр HTML-формы
html_path = authoring_assets['offline_form_ru']
display(FileLink(str(html_path)))
display(IFrame(src=str(html_path), width='100%', height='900px'))


## Как заполнять форму

### Рекомендуемый порядок
1. **Метаданные** — тема, creator_id, цель проверки.
2. **Добавление кейсов** — сначала `multimodal_hard`, затем `temporal_hard`, в конце немного `easy_control`.
3. **Контекст кейса** — статья, `paper_id`, страница/figure/table.
4. **creator_prompt** — одна короткая задача для участника A/B.
5. **creator_rationale** — почему кейс диагностический.
6. **review_focus** — 1–3 главных зоны внимания.
7. **expected_error_modes** — только реально ожидаемые ошибки.
8. **match.*`** — только если нужно более точное сопоставление.
9. **Черновик** — если хотите сделать паузу, нажмите **«Скачать черновик JSON»**, а при возвращении используйте **«Загрузить черновик JSON»**.
10. **Отправка** — после проверки скачайте итоговый JSON и при необходимости откройте **Google Форму** по ссылке внутри формы.

### Хороший `creator_prompt`
> Сравните, какой вариант лучше вытягивает численные значения из таблицы 2 и сохраняет правильную temporal scope.

### Хороший `creator_rationale`
> Кейс диагностический, потому что ключевой факт сидит в table-driven evidence, а baseline часто пропускает численные поля или теряет связь с временным интервалом.

### Когда использовать черновик
- когда набор заполняется в несколько сессий;
- когда хотите передать незавершённый вариант коллеге;
- когда боитесь потерять локальные изменения в браузере.

### Когда достаточно автосохранения
- когда вы работаете в одном и том же браузере и не закрываете черновик надолго.

### Когда нужен блок `match.*`
- когда вы заранее знаете стабильную сигнатуру candidate;
- когда хотите сузить сопоставление по title / premise / mechanism / time_scope;
- когда кейс потенциально неоднозначный.

### Чего лучше избегать
- длинных prompt на несколько задач сразу;
- слишком общих rationale без привязки к visual/temporal сложности;
- заполнения всех error modes подряд без необходимости;
- отправки данных в Google Форму до проверки статуса готовности кейсов.


## Что скачать в конце

После выполнения финальной ячейки notebook автоматически предложит скачать **два отдельных файла**:

1. `task3_ab_creator_offline_form_ru.html` — саму офлайн-форму для заполнения.
2. `task3_ab_creator_rest_files_ru.zip` — архив со всеми остальными материалами:
   - `task1_minimal_for_ab_bundle.zip`
   - `task3_ab_case_manifest.template.json`
   - `TASK3_AB_CREATOR_FORM_TUTORIAL_RU.md`
   - `task3_ab_creator_checklist_ru.md`
   - `README_RU.txt`

При паузе в работе дополнительно можно отдельно скачать:
- `task3_ab_case_manifest.draft.json`

После заполнения формы сохраните:
- `task3_ab_case_manifest.filled.json`

Если в вашей организации сбор идёт через веб-форму, откройте ссылку из офлайн-формы:
`https://forms.gle/h5RwEA8DsZh9pBAt8`

Итоговый JSON затем используется в CLI/Kaggle/DataSphere runner как case manifest.


In [ ]:

# @title Шаг 6. Автоскачивание HTML-формы и отдельного архива с остальными файлами
rest_zip = OUTPUT_DIR / 'task3_ab_creator_rest_files_ru.zip'

rest_files = [
    trajectory_zip,
    authoring_assets['manifest_template'],
    authoring_assets['tutorial_ru'],
    authoring_assets['checklist_ru'],
    OUTPUT_DIR / 'authoring_bundle_ru' / 'README_RU.txt',
]

with zipfile.ZipFile(rest_zip, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
    for p in rest_files:
        p = Path(p)
        if p.exists():
            zf.write(p, arcname=p.name)

display(Markdown("### Файлы готовы к скачиванию"))
print("HTML форма:", authoring_assets['offline_form_ru'])
display(FileLink(str(authoring_assets['offline_form_ru'])))
print("Архив с остальными файлами:", rest_zip)
display(FileLink(str(rest_zip)))

summary = {
    "offline_form_html": str(authoring_assets['offline_form_ru']),
    "rest_archive_zip": str(rest_zip),
    "trajectory_bundle_zip": str(trajectory_zip),
    "creator_manifest_template": str(authoring_assets['manifest_template']),
    "tutorial_ru": str(authoring_assets['tutorial_ru']),
    "checklist_ru": str(authoring_assets['checklist_ru']),
    "google_form_url": "https://forms.gle/h5RwEA8DsZh9pBAt8",
}
summary_path = OUTPUT_DIR / 'creator_download_summary.json'
summary_path.write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding='utf-8')
print("Сводка:", summary_path)
display(FileLink(str(summary_path)))

try:
    from google.colab import files
    print("Запускаю скачивание HTML формы...")
    files.download(str(authoring_assets['offline_form_ru']))
    print("Запускаю скачивание архива с остальными файлами...")
    files.download(str(rest_zip))
except Exception as e:
    print("Автоскачивание не запустилось автоматически.")
    print("Откройте ссылки выше и скачайте файлы вручную.")
    print("Причина:", repr(e))
